# 22-patient-accounting, per-patient accounting and slice-cap noise (Step 7.1)

Settings: **Accelerator None (CPU)**, **Internet On**, secret `kaggle_dpfl-braintumor` attached. No input dataset is needed.

Runs code commit `fdd02ebf7b510726c338519ca383103e45dafa21`: tests, then the per-patient epsilon of every training unit under the frozen record-level runs (central, natural and Dirichlet partitions), then the noise multiplier of every Phase 7.4 slice-cap arm. Pushes `results/tables/week13_patient_eps*` and `configs/sigma_v2.csv`. Uses no GPU quota; about one to two hours of CPU.

Generated by `scripts/make_launcher.py --kind accounting`, edit that script rather than this notebook.

In [ ]:
import os, sys, glob, subprocess, datetime, shutil
from kaggle_secrets import UserSecretsClient

USER, EMAIL, REPO = "shashiranjan-jnu", "shashi17_scs@jnu.ac.in", "dpfl-braintumor"
CODE_REF = "fdd02ebf7b510726c338519ca383103e45dafa21"
token = UserSecretsClient().get_secret("kaggle_dpfl-braintumor")
os.environ["GIT_PUSH_URL"] = f"https://{token}@github.com/{USER}/{REPO}.git"
CLEAN = f"https://github.com/{USER}/{REPO}.git"
CODE, RES = "/tmp/code", "/tmp/results"

def run(cmd, cwd=None, check=True):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(token, "***")
    if check and r.returncode != 0:
        raise RuntimeError(out[-3000:])
    return out

for d in (CODE, RES):
    shutil.rmtree(d, ignore_errors=True)
    run(["git", "clone", "-q", os.environ["GIT_PUSH_URL"], d])
    run(["git", "remote", "set-url", "origin", CLEAN], cwd=d)
run(["git", "checkout", "-q", CODE_REF], cwd=CODE)
run(["git", "config", "user.name", USER], cwd=RES)
run(["git", "config", "user.email", EMAIL], cwd=RES)
sys.path.insert(0, CODE)
from scripts.run_grid import push_run


print(run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements.txt"])[-500:])  # opacus etc.
import torch
TODAY = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d")
COMMIT = run(["git", "rev-parse", "HEAD"], cwd=CODE).strip()
for f in ("results/ncc/curated_pairs_ncc090.csv", "results/tables/week05_h5_mechanism.csv",
          "configs/grids/week07_wn_seeds.yaml"):
    assert os.path.exists(f"{RES}/{f}"), f"missing {f} in the results repository"
n_logits = len(glob.glob(f"{RES}/results/audit/*/logits.npz"))
print("code", CODE_REF, run(["git", "rev-parse", "--short", "HEAD"], cwd=CODE).strip(), "| logits files", n_logits)

In [ ]:
# Test suite on the Kaggle image, report pushed to results/tests/.
sys.path.insert(0, CODE)
from scripts.run_grid import push_run
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests", "-p", "no:cacheprovider"],
                   cwd=CODE, capture_output=True, text=True)
report = f"date {TODAY}\ncode {CODE_REF} {COMMIT}\ntorch {torch.__version__}\nreturncode {r.returncode}\n\n" + r.stdout[-6000:]
print(report[-2500:])
os.makedirs(f"{RES}/results/tests", exist_ok=True)
open(f"{RES}/results/tests/{TODAY}_{CODE_REF}_pytest.txt", "w").write(report)
print("push", push_run(RES, "results/tests", f"Test report {CODE_REF}"))
assert r.returncode == 0, "Tests failed, stop here and report"

In [ ]:
# Step 7.1 (E2.2): per-patient epsilon of the frozen record-level runs, and the sigma of every Phase 7.4 arm. CPU only.
R = f"{RES}/results"
T = f"{R}/tables"
print(run([sys.executable, "scripts/patient_accounting.py", "--results", R, "--out", f"{T}/week13_patient_eps"],
          cwd=CODE)[-8000:])
os.makedirs(f"{RES}/configs", exist_ok=True)
print(run([sys.executable, "scripts/calibrate_v2.py", "--out", f"{RES}/configs/sigma_v2.csv"], cwd=CODE)[-3000:])
print("push tables", push_run(RES, "results/tables", f"Per-patient accounting on Kaggle ({CODE_REF})"))
print("push sigma", push_run(RES, "configs", f"Phase 7.4 sigma on Kaggle ({CODE_REF})"))